# 🛡️ Guardrail Agent Testing Suite

This notebook tests the Guardrail Agent against various inputs to verify:
- PII patterns are correctly detected (SSN, credit cards, emails, etc.)
- Profanity blocklist works
- Hate speech blocklist works
- Safe inputs pass through without issues
- Scoring/threshold behavior is correct

Run this notebook after any changes to `backend/guardrails/rules.py`.

In [ ]:
import requests
import json

API_BASE = "http://localhost:8000"

def check(text: str) -> dict:
    resp = requests.post(f"{API_BASE}/guardrail/check?text={text}")
    return resp.json()


passed = 0
failed = 0

print("🧪 Guardrail Test Suite")
print("=" * 50)
print()

In [ ]:
# ── Safe Inputs (should PASS) ──
safe_tests = [
    ("What is machine learning?", "basic ML question"),
    ("Can you explain p-values?", "stats concept"),
    ("Review this Python function for me", "code review request"),
    ("How do I clean a CSV with missing values?", "data cleaning"),
    ("What is the best way to visualize this data?", "visualization question"),
]

print("✅ Testing SAFE inputs (should all PASS)...")
for text, label in safe_tests:
    result = check(text)
    if result["passed"]:
        print(f"  ✅ [{label}] PASSED (score={result['score']})")
        passed += 1
    else:
        print(f"  ❌ [{label}] BLOCKED — FALSE POSITIVE! (score={result['score']})")
        failed += 1
print()

In [ ]:
# ── PII Inputs (should BLOCK) ──
pii_tests = [
    ("My SSN is 123-45-6789", "SSN"),
    ("Call me at 555-123-4567", "phone number"),
    ("Email me at user@example.com", "email"),
    ("My card is 4111-1111-1111-1111", "credit card"),
    ("Server IP is 192.168.1.1", "IP address (low weight alone)"),
    ("My API key is sk-proj-abc123def456ghi789jkl012mno345pqr678stu", "API key"),
    ("Send BTC to 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa", "crypto wallet"),
]

print("🚫 Testing PII inputs (high score = should BLOCK)...")
for text, label in pii_tests:
    result = check(text)
    if not result["passed"]:
        print(f"  ✅ [{label}] BLOCKED (score={result['score']}) — {result['violations'][0]['category']}")
        passed += 1
    else:
        print(f"  ❌ [{label}] PASSED — FALSE NEGATIVE! (score={result['score']})")
        failed += 1
print()

In [ ]:
# ── Profanity / Hate Speech (should BLOCK) ──
content_tests = [
    ("This is fucking shit", "profanity"),
    ("You're an asshole", "insult"),
]

print("🚫 Testing PROFANITY inputs (should BLOCK)...")
for text, label in content_tests:
    result = check(text)
    if not result["passed"]:
        print(f"  ✅ [{label}] BLOCKED (score={result['score']})")
        passed += 1
    else:
        print(f"  ❌ [{label}] PASSED — FALSE NEGATIVE! (score={result['score']})")
        failed += 1
print()

In [ ]:
# ── Summary ──
total = passed + failed
print("=" * 50)
print(f"📊 Results: {passed}/{total} passed ({passed/total*100:.0f}%)")
if failed > 0:
    print(f"⚠️  {failed} tests failed! Review backend/guardrails/rules.py")
else:
    print("✅ All tests passed!")

## Edge Case Testing

Test edge cases like:
- Empty messages
- Very long messages
- Unicode/special characters
- Attempts to bypass the guardrail

In [ ]:
edge_cases = [
    ("a", "single character"),
    ("A" * 1000, "long message (1000 chars)"),
    ("héllo wörld ñoño", "unicode"),
    ("<script>alert('xss')</script>", "XSS attempt"),
    ("SELECT * FROM users; DROP TABLE users;", "SQL injection"),
]

print("🧪 Edge Cases")
for text, label in edge_cases:
    result = check(text)
    status = "PASSED" if result["passed"] else "BLOCKED"
    print(f"  [{label}] {status} (score={result['score']})")

## Try Your Own Test

In [ ]:
# Custom test — change the text below
custom_text = "Enter your test message here"

result = check(custom_text)
print(f"Text: {custom_text}")
print(json.dumps(result, indent=2))